# FlyWire Error Analysis: False Synapses Experiment

**One Kaggle Run = One Dataset + One Error Model + One Analysis Profile**

---
### How to run on Kaggle
1. Attach both datasets to this notebook:
   - `flywire-codebase` (uploaded from `flywire_codebase.zip`)
   - `flywire-all-datasets` (uploaded from `flywire_all_datasets.zip`)
2. In **Cell 3**, set `DATASET_NAME` to whichever connectome you want to run.
3. Click **Run All**.

### Kaggle Dataset Paths (fixed, no changes needed)
- Codebase : `/kaggle/input/datasets/jeet7771/flywire-codebase`
- Data      : `/kaggle/input/datasets/jeet7771/flywire-all-datasets`

In [ ]:
# Cell 1: Environment Setup & sys.path
# This MUST run before any framework imports.
# ============================================================
import os
import sys
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle/input')

# Exact Kaggle dataset mount paths for user jeet7771
KAGGLE_CODEBASE_PATH = Path('/kaggle/input/datasets/jeet7771/flywire-codebase')
KAGGLE_DATA_PATH     = Path('/kaggle/input/datasets/jeet7771/flywire-all-datasets')

if IS_KAGGLE:
    # --- Verify and add codebase to sys.path ---
    if not KAGGLE_CODEBASE_PATH.exists():
        raise FileNotFoundError(
            f'Codebase dataset not found at {KAGGLE_CODEBASE_PATH}\n'
            'Attach the "flywire-codebase" dataset to this notebook.'
        )
    sys.path.insert(0, str(KAGGLE_CODEBASE_PATH))
    print(f'[OK] Codebase path  : {KAGGLE_CODEBASE_PATH}')

    # --- Verify data dataset ---
    if not KAGGLE_DATA_PATH.exists():
        raise FileNotFoundError(
            f'Data dataset not found at {KAGGLE_DATA_PATH}\n'
            'Attach the "flywire-all-datasets" dataset to this notebook.'
        )
    print(f'[OK] Data path      : {KAGGLE_DATA_PATH}')
    print(f'[OK] Datasets found : {[d.name for d in KAGGLE_DATA_PATH.iterdir() if d.is_dir()]}')

else:
    # Local: codebase is the current working directory
    REPO_ROOT = Path(os.getcwd())
    if str(REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT))
    print(f'[OK] Running locally. Repo root: {REPO_ROOT}')

print(f'Environment: {"KAGGLE" if IS_KAGGLE else "LOCAL"}')

In [ ]:
# Cell 2: Framework Imports
import warnings
import pandas as pd

warnings.filterwarnings('ignore')

from core.experiment_runner import ExperimentRunner, ExperimentConfig
from modules.error_models import registry as error_registry
from modules.graph_analyses.analysis_registry import registry as analysis_registry
from modules.statistical_evaluation import StatisticalEvaluator
from core.export_manager import ExportManager
from modules.preprocessing import CandidateGenerator

print('All framework imports successful.')

In [ ]:
# ============================================================
# Cell 3: RUNTIME CONFIGURATION  <-- ONLY CELL YOU NEED TO EDIT
# ============================================================

# Which connectome to run.
# Options: "BANC" | "FAFB" | "MANC" | "MAOL" | "MCNS" | "TEST"
DATASET_NAME = "BANC"

# [LOCAL ONLY] Path to your raw dataset folder. Ignored on Kaggle.
LOCAL_DATASET_ROOT = "research_data/raw"

EXPERIMENT = {
    "metadata": {
        "experiment_name": f"FalseSynapses_{DATASET_NAME}",
        "author": "FlyWire Researcher",
        "description": "Topological degradation from simulated false-positive (false-synapse) errors.",
    },
    "error": {
        "name": "false_synapses",
        "rates": [
            0.00,    # 0%
            0.0025,  # 0.25%
            0.0050,  # 0.5%
            0.0075,  # 0.75%
            0.0100,  # 1%
            0.0200,  # 2%
            0.0500,  # 5%
            0.1000,  # 10%
            0.1500,  # 15%
            0.2000,  # 20%
        ],
        "random_seeds": [1, 2, 3, 4, 5],
    },
    "analysis": [
        "basic_structure",
        "degree_distribution",
        "pagerank",
        "assortativity",
        # "centrality",  # Temporarily disabled for runtime profiling
        # (Betweenness + Closeness are intractable on large graphs)
        "connected_components",
        "reciprocity",
    ],
    "export": {
        "create_zip": True,
        "save_statistics": True,
    },
}

OUTPUT_ROOT = Path("results") / DATASET_NAME / EXPERIMENT["error"]["name"]

print(f"Dataset Name    : {DATASET_NAME}")
print(f"Error Model     : {EXPERIMENT['error']['name']}")
print(f"Error Rates     : {EXPERIMENT['error']['rates']}")
print(f"Trials per Rate : {len(EXPERIMENT['error']['random_seeds'])}")
print(f"Output Root     : {OUTPUT_ROOT}")

In [ ]:
# Cell 4: Resolve Dataset Root
if IS_KAGGLE:
    DATASET_ROOT = str(KAGGLE_DATA_PATH)
else:
    DATASET_ROOT = '0-demodata' if DATASET_NAME.upper() == 'TEST' else LOCAL_DATASET_ROOT

print(f'DATASET_ROOT = {DATASET_ROOT}')

In [ ]:
# Cell 5: Verify Dataset Structure
from core.dataset_registry import DatasetRegistry, DatasetRegistryError

CONFIGS_ROOT = str(KAGGLE_CODEBASE_PATH / 'configs') if IS_KAGGLE else 'configs'

try:
    reg = DatasetRegistry(configs_root=CONFIGS_ROOT, dataset_root=DATASET_ROOT)
    resolved_dir = reg.resolve_dataset_dir(DATASET_NAME, DATASET_ROOT)
    print(f'[OK] Dataset "{DATASET_NAME}" verified.')
    print(f'     Resolved: {resolved_dir}')
except DatasetRegistryError as e:
    raise FileNotFoundError(
        f'Cannot resolve dataset "{DATASET_NAME}" in "{DATASET_ROOT}".\n'
        f'Expected a subfolder named {DATASET_NAME}_<version>/ or {DATASET_NAME}/.\n'
        f'Error: {e}'
    ) from e

In [ ]:
# Cell 6: Verify Registries
err_model = EXPERIMENT['error']['name']
print(f'Registered Error Models : {error_registry.list_names()}')
print(f'Registered Analyses     : {analysis_registry.list_names()}')

assert err_model in error_registry.list_names(), \
    f'Error model "{err_model}" not registered.'
missing = [a for a in EXPERIMENT['analysis'] if a not in analysis_registry.list_names()]
assert not missing, f'Analyses not registered: {missing}'

print('[OK] All required components registered. Ready to run.')

In [ ]:
# Cell 7: Candidate Generation (False Synapse specific — runs once)
# ============================================================
# Generates the ranked candidate edge table for the False Synapse
# error model.  This is a one-time preprocessing step per dataset.
# If the cache already exists, it is reused.
#
# NOTE: This cell loads the dataset independently from the
# ExperimentRunner below.  The dataset is loaded again during the
# first trial in Cell 9 — this is architecturally correct (each
# component owns its own pipeline) but adds ~5 min of loading time
# for large datasets like BANC.
# ============================================================

import time
from pathlib import Path

from core.data_loader import load_dataset
from core.graph_builder import GraphBuilder
from modules.preprocessing import preprocess_graph

print('Loading dataset for candidate generation...')
t_cand_start = time.perf_counter()

dataset = load_dataset(DATASET_NAME, DATASET_ROOT, configs_root=CONFIGS_ROOT)
graph = GraphBuilder().build(dataset)
prepared = preprocess_graph(
    graph,
    index_node_attrs=["top_region"],
    feature_config={
        "indegree": True,
        "outdegree": True,
        "pagerank": False,
        "reciprocal_ratio": False,
        "hub_neighbor_count": False,
        "two_hop_size": False,
    },
)
print(f'  Loaded {prepared.metadata.node_count} nodes, {prepared.metadata.edge_count} edges.')

# Determine cache path.
cache_dir = Path("research_data/cache/false_synapses")
cache_dir.mkdir(parents=True, exist_ok=True)
cache_path = cache_dir / f"candidates_{DATASET_NAME.lower()}.parquet"

if cache_path.exists():
    print(f'Loading cached candidate table from {cache_path} ...')
    import polars as pl
    cand_table = pl.read_parquet(str(cache_path))
    print(f'  Loaded {len(cand_table):,} candidates. Skipping generation.')
else:
    print(f'Generating candidate cache at {cache_path} ...')
    generator = CandidateGenerator(prepared)
    result_path = generator.generate(cache_path)
    elapsed = time.perf_counter() - t_cand_start
    print(f'  Done in {elapsed:.1f} s.')
    import polars as pl
    cand_table = pl.read_parquet(str(cache_path))
    print(f'  Generated {len(cand_table):,} candidate pairs.')

# Warn if no candidates were found.
if len(cand_table) == 0:
    print('[WARN] No candidates found. This may happen on small datasets.')
    print('       Every trial will add 0 false edges (k = round(error_rate * edge_count) = 0).')
    print('       To proceed anyway, the experiment will run baseline analyses only.')

# Release the one-off PreparedGraph — not needed until experiments.
del prepared, graph, dataset

In [ ]:
# Cell 8: Run Experiments
runner = ExperimentRunner(analysis_registry, error_registry)
results_per_rate = {}

for err_rate in EXPERIMENT['error']['rates']:
    rate_str = f"{err_rate * 100:g}".replace('.', '_') + "_percent"
    results_per_rate[err_rate] = []

    for trial, seed in enumerate(EXPERIMENT['error']['random_seeds'], 1):
        print(f'\n{"="*50}')
        print(f'  Dataset    : {DATASET_NAME}')
        print(f'  Error Rate : {err_rate * 100:g}%')
        print(f'  Trial      : {trial} / {len(EXPERIMENT["error"]["random_seeds"])}')
        print(f'  Seed       : {seed}')
        print(f'{"="*50}')

        trial_out = OUTPUT_ROOT / rate_str / f'trial_{trial:03d}'

        config = ExperimentConfig(
            dataset_name=DATASET_NAME,
            dataset_root=str(DATASET_ROOT),
            configs_root=CONFIGS_ROOT,
            error_model_name=err_model,
            error_model_config={
                'error_rate': err_rate,
                'candidate_cache_path': str(cache_path),
            },
            analysis_names=EXPERIMENT['analysis'],
            preprocessing_config={'features': {'degree': True, 'synapse_counts': True}},
            seed=seed,
            output_root=str(trial_out) if EXPERIMENT['export']['save_statistics'] else None,
            create_zip=EXPERIMENT['export']['create_zip'],
            extra={'metadata': EXPERIMENT['metadata']},
        )

        res = runner.run(config)
        results_per_rate[err_rate].append(res)

        if res.succeeded:
            print(f'  --> Success! Runtime: {res.runtime_seconds:.2f}s')
        else:
            print(f'  --> FAILED!  Errors: {res.errors}')

print('\nAll trials complete.')

In [ ]:
# Cell 9: Statistical Evaluation
evaluator = StatisticalEvaluator()
aggregated_stats_by_rate = {}

baseline_runs = [r for r in results_per_rate.get(0.00, []) if r.succeeded]
if not baseline_runs:
    raise RuntimeError('No successful baseline (0%) runs. Cannot evaluate.')

for err_rate, run_results in results_per_rate.items():
    successful = [r for r in run_results if r.succeeded]
    if successful:
        eval_result = evaluator.evaluate(baseline_runs, successful)
        aggregated_stats_by_rate[err_rate] = eval_result
        print(f'Evaluated {err_rate*100:g}%  -> {len(successful)} successful trials')
    else:
        print(f'Skipped   {err_rate*100:g}%  -> 0 successful trials')

print('\nStatistical evaluation complete.')

In [ ]:
# Cell 10: Export Presentation Layer
# Plots -> results/<DATASET>/false_synapses/presentation/plots/
ExportManager().export_presentation(
    results_by_rate=aggregated_stats_by_rate,
    output_root=OUTPUT_ROOT,
    metadata=EXPERIMENT['metadata'],
)
print(f'Presentation exported to : {OUTPUT_ROOT / "presentation"}')
print(f'Plots saved to           : {OUTPUT_ROOT / "presentation" / "plots"}')

In [ ]:
# Cell 11: Summary Table
print('\n' + '=' * 60)
print('  SUMMARY TABLE — FALSE SYNAPSE EXPERIMENT')
print('=' * 60)

for err_rate in sorted(aggregated_stats_by_rate.keys()):
    ev = aggregated_stats_by_rate[err_rate]
    print(f'\n  Error Rate: {err_rate*100:g}%')
    print(f'  {"-" * 40}')
    for analysis_name, metrics in ev.metrics.items():
        print(f'  Analysis: {analysis_name}')
        df = pd.DataFrame([
            {
                'Metric': m_name,
                'Baseline Mean': m_dict.get('baseline_mean', 0),
                'Perturbed Mean': m_dict.get('perturbed_mean', 0),
                'Std': m_dict.get('std', 0),
                'CI Lower': m_dict.get('ci_lower', 0),
                'CI Upper': m_dict.get('ci_upper', 0),
            }
            for m_name, m_dict in metrics.items()
        ])
        if not df.empty:
            display(df)
        break  # only first analysis for brevity

In [ ]:
# Cell 12: Final Output Summary
print('\n' + '=' * 60)
print('  FALSE SYNAPSE EXPERIMENT — COMPLETE')
print('=' * 60)
print(f'  Dataset     : {DATASET_NAME}')
print(f'  Error Model : {err_model}')
print(f'  Output      : {OUTPUT_ROOT}')
print(f'  Rates       : {len([r for r in aggregated_stats_by_rate])} rates evaluated')
total_trials = sum(len(v) for v in results_per_rate.values())
print(f'  Trials      : {total_trials}')
print()
print('  The results are ready for review in the output directory above.')
print('  All analyses, CSVs, JSON exports, and plots have been saved.')
print('=' * 60)


In [ ]:
# Cell 13: Zip Results for Kaggle Download
import zipfile
from pathlib import Path

if OUTPUT_ROOT.exists():
    zip_path = Path(f'{OUTPUT_ROOT.parent}_false_synapses_results.zip')
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for fpath in OUTPUT_ROOT.rglob('*'):
            if fpath.is_file():
                zf.write(fpath, arcname=fpath.relative_to(OUTPUT_ROOT.parent))
    print(f'[OK] Results archived to {zip_path}')
else:
    print('[SKIP] Output directory not found. Nothing to zip.')